# Sync RAS Curations

Syncs verb-based RAS curations (add/remove institution_ids) from the users Heroku Postgres database to a local Databricks table.

**Source**: `openalex_users.public.ras_institution_curations` (Heroku Postgres view, migration 067)
**Target**: `openalex.institutions.ras_curations` (Delta table)

Curations use verb-based semantics:
- `action='add'`: Include this institution_id even if model didn't predict it
- `action='remove'`: Exclude this institution_id even if model predicted it

**View contract**: one row per (raw_affiliation_string, institution_id) *pair* carrying the winning action — NOT one row per curation row, unlike the other `_curations` views. Latest-action-wins (the oxjob #582 toggle fix) lives in the view, in the same repo as the write path; rationale in the migration 067 header. This notebook fans each pair out over its mojibake equivalence class (oxjob #801 — works display the repaired string but stay keyed on the original bytes; see `affiliation_strings_repair`), re-resolves latest-action-wins per (string, institution), then does the array pivot; the sync is insert/update-only (RAS curations are an append-only event log — undo = the opposite action, and row deletion is blocked at the API).

Timestamps on the target:
- `latest_curation_at` = MAX(source `created`) per string — when a curator last acted (the honest change signal)
- `updated_datetime` = when this sync last ran (rewritten every night)


## Sync curations from users DB


In [ ]:
%sql
-- Preview what will be synced. The view already resolved latest-action-wins
-- per (string, institution) pair (migration 067, oxjob #582/#684); this fans
-- each pair out over its mojibake equivalence class (oxjob #801) and does the
-- scalar -> array pivot (arrays stay on this side — PG arrays over Lakehouse
-- Federation are fragile).
WITH src AS (
  SELECT raw_affiliation_string, institution_id, action, created
  FROM openalex_users.public.ras_institution_curations
),
-- oxjob #801: fan each curation out over its mojibake EQUIVALENCE CLASS.
-- Works display the repaired string R (CreateWorkAuthorships) but stay keyed
-- internally on the mojibake bytes M, so a curation a curator submits against
-- the string they SEE (R) must also land on every M with repair(M) = R, and a
-- curation on M must land on R and its sibling variants. The class is derived
-- from affiliation_strings_repair (raw M -> repaired R; PrepareAffiliationStrings).
-- Gated to curations created on/after RAS_CLASS_EXPANSION_SINCE (the display
-- repair's ship date): expanding OLDER curations across their class would move
-- institution links on 667 works (measured 2026-08-18) — a deliberate
-- attribution change that needs Jason's sign-off, not a side effect of an
-- encoding fix. Lifting the gate = deleting the WHERE below.
class_key AS (
  SELECT DISTINCT
    s.raw_affiliation_string AS k,
    COALESCE(rep.repaired_string, s.raw_affiliation_string) AS ck
  FROM src s
  LEFT JOIN openalex.institutions.affiliation_strings_repair rep
    ON rep.raw_affiliation_string = s.raw_affiliation_string
),
class_members AS (
  SELECT k, ck AS member FROM class_key
  UNION
  SELECT ck.k, rep.raw_affiliation_string AS member
  FROM class_key ck
  JOIN openalex.institutions.affiliation_strings_repair rep
    ON rep.repaired_string = ck.ck
),
expanded AS (
  SELECT raw_affiliation_string, institution_id, action, created FROM src
  UNION ALL
  SELECT m.member AS raw_affiliation_string, s.institution_id, s.action, s.created
  FROM src s
  JOIN class_members m
    ON m.k = s.raw_affiliation_string
   AND m.member <> s.raw_affiliation_string
  WHERE s.created >= TIMESTAMP '2026-08-19 00:00:00'  -- RAS_CLASS_EXPANSION_SINCE (UTC)
),
-- Re-resolve latest-action-wins at the (member string, institution) level: a
-- string can now receive actions from several class keys (same semantics as
-- the migration 067 view, one level up).
resolved AS (
  SELECT
    raw_affiliation_string,
    institution_id,
    MAX_BY(action, created) AS action,
    MAX(created) AS created
  FROM expanded
  GROUP BY raw_affiliation_string, institution_id
)
SELECT
  raw_affiliation_string,
  FILTER(
    ARRAY_AGG(CASE WHEN action = 'add' THEN institution_id END),
    x -> x IS NOT NULL
  ) AS curated_add_ids,
  FILTER(
    ARRAY_AGG(CASE WHEN action = 'remove' THEN institution_id END),
    x -> x IS NOT NULL
  ) AS curated_remove_ids,
  MAX(created) AS latest_curation_at,
  COUNT(*) AS num_pairs
FROM resolved
GROUP BY raw_affiliation_string
LIMIT 100


In [ ]:
%sql
-- MERGE curations into local table (inserts + updates only — deliberately NO
-- NOT MATCHED BY SOURCE DELETE). The curations log is append-only for RAS:
-- undo = submit the opposite action (latest-action-wins in the source view,
-- migration 067); row deletion is blocked at the API. A row present here but
-- absent from the view would mean an out-of-band PG delete — preserved here
-- rather than silently un-applied.
MERGE INTO openalex.institutions.ras_curations AS target
USING (
  WITH src AS (
    SELECT raw_affiliation_string, institution_id, action, created
    FROM openalex_users.public.ras_institution_curations
  ),
  -- oxjob #801: fan each curation out over its mojibake EQUIVALENCE CLASS.
  -- Works display the repaired string R (CreateWorkAuthorships) but stay keyed
  -- internally on the mojibake bytes M, so a curation a curator submits against
  -- the string they SEE (R) must also land on every M with repair(M) = R, and a
  -- curation on M must land on R and its sibling variants. The class is derived
  -- from affiliation_strings_repair (raw M -> repaired R; PrepareAffiliationStrings).
  -- Gated to curations created on/after RAS_CLASS_EXPANSION_SINCE (the display
  -- repair's ship date): expanding OLDER curations across their class would move
  -- institution links on 667 works (measured 2026-08-18) — a deliberate
  -- attribution change that needs Jason's sign-off, not a side effect of an
  -- encoding fix. Lifting the gate = deleting the WHERE below.
  class_key AS (
    SELECT DISTINCT
      s.raw_affiliation_string AS k,
      COALESCE(rep.repaired_string, s.raw_affiliation_string) AS ck
    FROM src s
    LEFT JOIN openalex.institutions.affiliation_strings_repair rep
      ON rep.raw_affiliation_string = s.raw_affiliation_string
  ),
  class_members AS (
    SELECT k, ck AS member FROM class_key
    UNION
    SELECT ck.k, rep.raw_affiliation_string AS member
    FROM class_key ck
    JOIN openalex.institutions.affiliation_strings_repair rep
      ON rep.repaired_string = ck.ck
  ),
  expanded AS (
    SELECT raw_affiliation_string, institution_id, action, created FROM src
    UNION ALL
    SELECT m.member AS raw_affiliation_string, s.institution_id, s.action, s.created
    FROM src s
    JOIN class_members m
      ON m.k = s.raw_affiliation_string
     AND m.member <> s.raw_affiliation_string
    WHERE s.created >= TIMESTAMP '2026-08-19 00:00:00'  -- RAS_CLASS_EXPANSION_SINCE (UTC)
  ),
  -- Re-resolve latest-action-wins at the (member string, institution) level: a
  -- string can now receive actions from several class keys (same semantics as
  -- the migration 067 view, one level up).
  resolved AS (
    SELECT
      raw_affiliation_string,
      institution_id,
      MAX_BY(action, created) AS action,
      MAX(created) AS created
    FROM expanded
    GROUP BY raw_affiliation_string, institution_id
  )
  SELECT
    raw_affiliation_string,
    FILTER(
      ARRAY_AGG(CASE WHEN action = 'add' THEN institution_id END),
      x -> x IS NOT NULL
    ) AS curated_add_ids,
    FILTER(
      ARRAY_AGG(CASE WHEN action = 'remove' THEN institution_id END),
      x -> x IS NOT NULL
    ) AS curated_remove_ids,
    MAX(created) AS latest_curation_at,
    CURRENT_TIMESTAMP() AS updated_datetime
  FROM resolved
  GROUP BY raw_affiliation_string
) AS source
ON target.raw_affiliation_string = source.raw_affiliation_string
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *


## Verify sync results


In [ ]:
%sql
-- Check local curations table
SELECT 
  COUNT(*) AS total_curated_ras,
  SUM(SIZE(curated_add_ids)) AS total_adds,
  SUM(SIZE(curated_remove_ids)) AS total_removes,
  MAX(latest_curation_at) AS last_curator_action,
  MAX(updated_datetime) AS last_sync
FROM openalex.institutions.ras_curations


In [ ]:
%sql
-- Sample of recently curated RAS
SELECT * FROM openalex.institutions.ras_curations
ORDER BY latest_curation_at DESC
LIMIT 10
